In [1]:
!pip install wfdb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 37.9 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.


In [2]:

"""
Universal ECG Dataset Loader for AFib Research
A truly generalized approach that can adapt to any ECG dataset format
"""

import os
import numpy as np
import pandas as pd
import wfdb
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union
import json
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ============================================================================
# STEP 1: DATASET DETECTOR - Automatically detects dataset format
# ============================================================================
class DatasetDetector:
    """
    Intelligently detects the format and structure of any ECG dataset.
    This is the KEY to making the loader work with unknown datasets.
    """

    def __init__(self, data_path: str):
        self.data_path = Path(data_path)
        self.structure = {}

    def analyze_directory(self) -> Dict:
        """
        Analyzes directory structure and file types to understand the dataset.

        Returns a dictionary describing:
        - File formats found (.dat, .hea, .csv, .mat, .edf, etc.)
        - Directory structure
        - Potential metadata files
        - Number of records
        """
        print(f"\n{'='*60}")
        print(f"ANALYZING DATASET AT: {self.data_path}")
        print(f"{'='*60}\n")

        analysis = {
            'path': str(self.data_path),
            'file_extensions': {},
            'subdirectories': [],
            'metadata_files': [],
            'record_files': [],
            'format_type': None
        }

        # Scan all files
        for file_path in self.data_path.rglob('*'):
            if file_path.is_file():
                ext = file_path.suffix.lower()

                # Count file types
                if ext:
                    analysis['file_extensions'][ext] = analysis['file_extensions'].get(ext, 0) + 1

                # Identify metadata files
                if ext in ['.csv', '.json', '.txt', '.xlsx'] or 'metadata' in file_path.name.lower():
                    analysis['metadata_files'].append(str(file_path.relative_to(self.data_path)))

                # Identify record files (signal data)
                if ext in ['.dat', '.mat', '.edf', '.npy', '.npz']:
                    analysis['record_files'].append(str(file_path.relative_to(self.data_path)))

        # Identify subdirectories
        for item in self.data_path.iterdir():
            if item.is_dir():
                analysis['subdirectories'].append(item.name)

        # Determine format type
        analysis['format_type'] = self._determine_format(analysis)

        # Print analysis
        self._print_analysis(analysis)

        return analysis

    def _determine_format(self, analysis: Dict) -> str:
        """
        Determines the dataset format based on file extensions.

        Common formats:
        - WFDB: .hea + .dat files (MIT-BIH, PhysioNet databases)
        - EDF: .edf files (European Data Format)
        - MATLAB: .mat files
        - CSV: .csv files with time series
        - NumPy: .npy or .npz files
        """
        exts = analysis['file_extensions']

        if '.hea' in exts and '.dat' in exts:
            return 'WFDB'
        elif '.edf' in exts:
            return 'EDF'
        elif '.mat' in exts:
            return 'MATLAB'
        elif '.csv' in exts:
            return 'CSV'
        elif '.npy' in exts or '.npz' in exts:
            return 'NUMPY'
        else:
            return 'UNKNOWN'

    def _print_analysis(self, analysis: Dict):
        """Pretty print the analysis results."""
        print(f"📁 Directory Structure:")
        print(f"   Root: {analysis['path']}")
        if analysis['subdirectories']:
            print(f"   Subdirectories: {', '.join(analysis['subdirectories'][:5])}")
            if len(analysis['subdirectories']) > 5:
                print(f"   ... and {len(analysis['subdirectories']) - 5} more")

        print(f"\n📄 File Types Found:")
        for ext, count in sorted(analysis['file_extensions'].items()):
            print(f"   {ext}: {count} files")

        print(f"\n📊 Metadata Files:")
        if analysis['metadata_files']:
            for mf in analysis['metadata_files'][:5]:
                print(f"   - {mf}")
            if len(analysis['metadata_files']) > 5:
                print(f"   ... and {len(analysis['metadata_files']) - 5} more")
        else:
            print(f"   No metadata files found")

        print(f"\n🔍 Detected Format: {analysis['format_type']}")
        print(f"   Total record files: {len(analysis['record_files'])}")
        print(f"\n{'='*60}\n")


In [4]:
# ============================================================================
# STEP 2: FORMAT-SPECIFIC READERS - Handles different file formats
# ============================================================================
class FormatReader:
    """
    Contains readers for different ECG data formats.
    Easy to extend with new format readers.
    """

    @staticmethod
    def read_wfdb(file_path: Path, record_name: str) -> Dict:
        """
        Read WFDB format (.hea + .dat files).
        Used by: MIT-BIH, PhysioNet databases
        """
        try:
            record = wfdb.rdrecord(str(file_path / record_name))

            # Try to load annotations
            annotation = None
            for ext in ['atr', 'qrs', 'ann']:
                try:
                    annotation = wfdb.rdann(str(file_path / record_name), ext)
                    break
                except:
                    continue

            data = {
                'signal': record.p_signal,
                'fs': record.fs,  # Now with comment explaining it's from .hea
                'channels': record.sig_name,  # Now with comment
                'n_channels': record.n_sig,  # NEW: Number of channels
                'units': record.units,
                'duration': len(record.p_signal) / record.fs,
                'n_samples': record.sig_len,  # NEW: Total samples
                'format': 'WFDB',
                'base_time': record.base_time,  # NEW: Timing info
                'base_date': record.base_date,  # NEW: Date info
                'comments': record.comments  # NEW: Any comments
            }

            if annotation:
                data['annotations'] = {
                    'samples': annotation.sample,
                    'symbols': annotation.symbol,
                    'aux_notes': getattr(annotation, 'aux_note', None)
                }

            return data
        except Exception as e:
            raise Exception(f"Error reading WFDB file: {str(e)}")

    @staticmethod
    def read_edf(file_path: Path) -> Dict:
        """
        Read EDF format files.
        Requires: pyedflib (install with: pip install pyedflib)
        """
        try:
            import pyedflib

            f = pyedflib.EdfReader(str(file_path))
            n_signals = f.signals_in_file

            signals = []
            channels = []
            fs_list = []

            for i in range(n_signals):
                signals.append(f.readSignal(i))
                channels.append(f.getLabel(i))
                fs_list.append(f.getSampleFrequency(i))

            signal = np.column_stack(signals)

            data = {
                'signal': signal,
                'fs': fs_list[0],  # Assuming same sampling rate
                'channels': channels,
                'duration': len(signal) / fs_list[0],
                'format': 'EDF'
            }

            f.close()
            return data

        except ImportError:
            raise ImportError("pyedflib not installed. Install with: pip install pyedflib")
        except Exception as e:
            raise Exception(f"Error reading EDF file: {str(e)}")

    @staticmethod
    def read_matlab(file_path: Path) -> Dict:
        """
        Read MATLAB .mat files.
        Requires: scipy
        """
        try:
            from scipy.io import loadmat

            mat_data = loadmat(str(file_path))

            # Try to find signal data (common variable names)
            signal_keys = ['signal', 'data', 'ecg', 'val', 'values']
            signal = None

            for key in signal_keys:
                if key in mat_data:
                    signal = mat_data[key]
                    break

            if signal is None:
                # Take the largest array that's not metadata
                for key, value in mat_data.items():
                    if not key.startswith('__') and isinstance(value, np.ndarray):
                        if value.ndim >= 1:
                            signal = value
                            break

            # Try to find sampling frequency
            fs_keys = ['fs', 'freq', 'frequency', 'sampling_rate', 'Fs']
            fs = 250  # Default

            for key in fs_keys:
                if key in mat_data:
                    fs = float(mat_data[key].flatten()[0])
                    break

            data = {
                'signal': signal,
                'fs': fs,
                'channels': [f'Channel_{i}' for i in range(signal.shape[1])] if signal.ndim > 1 else ['Channel_0'],
                'duration': len(signal) / fs,
                'format': 'MATLAB',
                'raw_keys': [k for k in mat_data.keys() if not k.startswith('__')]
            }

            return data

        except Exception as e:
            raise Exception(f"Error reading MATLAB file: {str(e)}")

    @staticmethod
    def read_csv(file_path: Path) -> Dict:
        """
        Read CSV format ECG data.
        Assumes: columns are channels, rows are samples
        """
        try:
            df = pd.read_csv(file_path)

            # Remove non-numeric columns (likely labels/annotations)
            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            signal = df[numeric_cols].values

            # Try to infer sampling frequency from time column
            fs = 250  # Default
            if 'time' in df.columns.str.lower() or 't' in df.columns:
                time_col = [c for c in df.columns if c.lower() in ['time', 't']][0]
                time_diff = df[time_col].diff().median()
                if time_diff > 0:
                    fs = 1 / time_diff

            data = {
                'signal': signal,
                'fs': fs,
                'channels': numeric_cols,
                'duration': len(signal) / fs,
                'format': 'CSV',
                'all_columns': df.columns.tolist()
            }

            return data

        except Exception as e:
            raise Exception(f"Error reading CSV file: {str(e)}")

    @staticmethod
    def read_numpy(file_path: Path) -> Dict:
        """
        Read NumPy .npy or .npz files.
        """
        try:
            if file_path.suffix == '.npz':
                data_file = np.load(str(file_path))
                # Get the first array
                signal = data_file[data_file.files[0]]
                available_keys = data_file.files
            else:
                signal = np.load(str(file_path))
                available_keys = None

            data = {
                'signal': signal,
                'fs': 250,  # Default - needs to be specified
                'channels': [f'Channel_{i}' for i in range(signal.shape[1])] if signal.ndim > 1 else ['Channel_0'],
                'duration': len(signal) / 250,
                'format': 'NUMPY'
            }

            if available_keys:
                data['available_keys'] = available_keys

            return data

        except Exception as e:
            raise Exception(f"Error reading NumPy file: {str(e)}")

In [5]:
from pathlib import Path
from typing import List, Dict, Optional
import numpy as np

class UniversalECGLoader:
    """
    Robust ECG loader that handles problematic records gracefully.

    Usage:
        loader = UniversalECGLoader('path/to/dataset')
        records = loader.get_valid_records()  # Only returns loadable records
        data = loader.load_record(records[0])
    """

    def __init__(self, data_path: str, auto_detect: bool = True):
        """
        Initialize the universal loader.

        Args:
            data_path: Path to the dataset directory
            auto_detect: If True, automatically detect dataset format
        """
        self.data_path = Path(data_path)

        if not self.data_path.exists():
            raise FileNotFoundError(f"Path does not exist: {data_path}")

        # Assuming you have DatasetDetector and FormatReader classes
        # from your previous code
        self.detector = DatasetDetector(data_path)
        self.reader = FormatReader()

        if auto_detect:
            self.analysis = self.detector.analyze_directory()
            self.format_type = self.analysis['format_type']
        else:
            self.format_type = None

        # Track problematic records
        self.failed_records = []
        self.valid_records_cache = None

    def get_records(self) -> List[str]:
        """
        Get list of all records in the dataset.
        Returns record identifiers (file names without extensions).
        """
        records = []

        if self.format_type == 'WFDB':
            # Find all .hea files
            for hea_file in self.data_path.rglob('*.hea'):
                # Skip backup files
                if not hea_file.name.endswith('-'):
                    records.append(hea_file.stem)

        elif self.format_type in ['EDF', 'MATLAB', 'CSV', 'NUMPY']:
            # Find all data files
            ext_map = {
                'EDF': '.edf',
                'MATLAB': '.mat',
                'CSV': '.csv',
                'NUMPY': '.npy'
            }
            ext = ext_map.get(self.format_type, '')

            for data_file in self.data_path.rglob(f'*{ext}'):
                # Skip metadata files
                if 'metadata' not in data_file.name.lower():
                    records.append(str(data_file.relative_to(self.data_path)))

        print(f"✅ Found {len(records)} records in {self.format_type} format")
        return sorted(records)

    def get_valid_records(self, force_check: bool = False) -> List[str]:
        """
        Get list of only valid (loadable) records by checking each one.
        This is slower but ensures all returned records can be loaded.

        Args:
            force_check: If True, re-check all records even if cached

        Returns:
            List of valid record identifiers
        """
        if self.valid_records_cache is not None and not force_check:
            return self.valid_records_cache

        all_records = self.get_records()
        valid_records = []
        failed_records = []

        print(f"\n🔍 Validating {len(all_records)} records...")

        for i, rec in enumerate(all_records, 1):
            try:
                # Try loading just the header or minimal data
                _ = self.load_record(rec)
                valid_records.append(rec)

                if i % 10 == 0:
                    print(f"   Progress: {i}/{len(all_records)} ({len(valid_records)} valid)")

            except Exception as e:
                failed_records.append((rec, str(e)))
                if i % 10 == 0:
                    print(f"   Progress: {i}/{len(all_records)} ({len(valid_records)} valid)")

        # Cache results
        self.valid_records_cache = valid_records
        self.failed_records = failed_records

        print(f"\n✅ Found {len(valid_records)} valid records")

        if failed_records:
            print(f"⚠️  Skipped {len(failed_records)} problematic records:")
            for rec, error in failed_records[:5]:  # Show first 5
                print(f"   • {rec}: {error[:60]}...")
            if len(failed_records) > 5:
                print(f"   ... and {len(failed_records) - 5} more")

        return valid_records

    def load_record(self, record_id: str, **kwargs) -> Dict:
        """
        Load a single ECG record with error handling.

        Args:
            record_id: Record identifier (from get_records())
            **kwargs: Additional arguments passed to format-specific reader

        Returns:
            Dictionary with standardized format
        """
        try:
            if self.format_type == 'WFDB':
                return self.reader.read_wfdb(self.data_path, record_id)

            elif self.format_type == 'EDF':
                file_path = self.data_path / record_id
                return self.reader.read_edf(file_path)

            elif self.format_type == 'MATLAB':
                file_path = self.data_path / record_id
                return self.reader.read_matlab(file_path)

            elif self.format_type == 'CSV':
                file_path = self.data_path / record_id
                return self.reader.read_csv(file_path)

            elif self.format_type == 'NUMPY':
                file_path = self.data_path / record_id
                return self.reader.read_numpy(file_path)

            else:
                raise ValueError(f"Unknown format: {self.format_type}")

        except Exception as e:
            # Re-raise with more context
            raise Exception(f"Failed to load record '{record_id}': {str(e)}")

    def load_all(self, max_records: Optional[int] = None, use_valid_only: bool = True) -> List[Dict]:
        """
        Load all records from the dataset.

        Args:
            max_records: Maximum number of records to load (None = all)
            use_valid_only: If True, only load pre-validated records

        Returns:
            List of loaded records
        """
        if use_valid_only:
            records = self.get_valid_records()
        else:
            records = self.get_records()

        if max_records:
            records = records[:max_records]

        print(f"\n🔄 Loading {len(records)} records...")
        all_data = []

        for i, rec in enumerate(records, 1):
            try:
                data = self.load_record(rec)
                data['record_id'] = rec
                all_data.append(data)

                if i % 10 == 0:
                    print(f"   Progress: {i}/{len(records)}")

            except Exception as e:
                print(f"   ⚠️  Failed to load {rec}: {str(e)[:60]}...")
                continue

        print(f"\n✅ Successfully loaded {len(all_data)}/{len(records)} records\n")
        return all_data

    def get_dataset_info(self, sample_size: int = 20) -> Dict:
        """
        Get comprehensive information about the loaded dataset.

        Args:
            sample_size: Number of records to sample for analysis
        """
        # Use valid records only
        records = self.get_valid_records()

        if not records:
            return {"error": "No valid records found"}

        print(f"\n📊 Analyzing dataset characteristics...")

        info = {
            'total_records': len(self.get_records()),
            'valid_records': len(records),
            'failed_records': len(self.failed_records),
            'sampling_frequencies': {},
            'lead_configurations': {},
            'duration_stats': {
                'min': float('inf'),
                'max': 0,
                'durations': []
            },
            'formats': {},
            'sample_records': []
        }

        # Sample records for analysis
        sample_size = min(sample_size, len(records))
        # Take evenly distributed samples
        step = max(1, len(records) // sample_size)
        sample_records = records[::step][:sample_size]

        for rec in sample_records:
            try:
                data = self.load_record(rec)

                # Track sampling frequency
                fs = data['fs']
                info['sampling_frequencies'][fs] = info['sampling_frequencies'].get(fs, 0) + 1

                # Track lead configuration
                num_leads = len(data['channels'])
                lead_config = f"{num_leads}-lead"
                info['lead_configurations'][lead_config] = info['lead_configurations'].get(lead_config, 0) + 1

                # Track duration
                duration = data['duration']
                info['duration_stats']['durations'].append(duration)
                info['duration_stats']['min'] = min(info['duration_stats']['min'], duration)
                info['duration_stats']['max'] = max(info['duration_stats']['max'], duration)

                # Track format
                fmt = data['format']
                info['formats'][fmt] = info['formats'].get(fmt, 0) + 1

                # Store sample record info
                info['sample_records'].append({
                    'record_id': rec,
                    'fs': fs,
                    'leads': data['channels'],
                    'duration': duration,
                    'signal_shape': data['signal'].shape
                })

            except Exception as e:
                print(f"   ⚠️  Could not analyze {rec}: {str(e)[:40]}...")
                continue

        # Calculate statistics
        if info['duration_stats']['durations']:
            info['duration_stats']['mean'] = np.mean(info['duration_stats']['durations'])
            info['duration_stats']['median'] = np.median(info['duration_stats']['durations'])

        # Print summary
        self._print_dataset_info(info)

        return info

    def _print_dataset_info(self, info: Dict):
        """Pretty print dataset information."""
        print(f"\n{'='*60}")
        print(f"DATASET SUMMARY")
        print(f"{'='*60}\n")

        print(f"📈 Total Records: {info['total_records']}")
        print(f"✅ Valid Records: {info['valid_records']}")
        if info['failed_records'] > 0:
            print(f"⚠️  Failed Records: {info['failed_records']}")

        print(f"\n🔊 Sampling Frequencies Found:")
        for fs, count in sorted(info['sampling_frequencies'].items()):
            print(f"   • {fs} Hz: {count} record(s)")

        print(f"\n📡 Lead Configurations:")
        for config, count in sorted(info['lead_configurations'].items()):
            print(f"   • {config}: {count} record(s)")

        print(f"\n⏱️  Duration Statistics:")
        if info['duration_stats']['durations']:
            print(f"   • Min: {info['duration_stats']['min']:.2f} seconds")
            print(f"   • Max: {info['duration_stats']['max']:.2f} seconds")
            print(f"   • Mean: {info['duration_stats']['mean']:.2f} seconds")
            print(f"   • Median: {info['duration_stats']['median']:.2f} seconds")

        print(f"\n📝 Sample Records:")
        for sample in info['sample_records'][:3]:
            print(f"   • {sample['record_id']}:")
            print(f"     - Frequency: {sample['fs']} Hz")
            print(f"     - Leads: {', '.join(sample['leads'][:3])}{'...' if len(sample['leads']) > 3 else ''}")
            print(f"     - Duration: {sample['duration']:.2f}s")
            print(f"     - Shape: {sample['signal_shape']}")

        print(f"\n{'='*60}\n")

In [6]:
# Initialize loader
loader = UniversalECGLoader('/content/drive/MyDrive/AF/files')

# Get only valid records (automatically filters out problematic ones)
valid_records = loader.get_valid_records()
print(f"Valid records: {valid_records}")

# Load a single record
if valid_records:
    data = loader.load_record(valid_records[0])
    print(f"Signal shape: {data['signal'].shape}")
    print(f"Sampling rate: {data['fs']} Hz")
    print(f"Channels: {data['channels']}")

# Get dataset information
info = loader.get_dataset_info()

# Load all valid records
all_data = loader.load_all(max_records=10)  # Load first 10


ANALYZING DATASET AT: /content/drive/MyDrive/AF/files

📁 Directory Structure:
   Root: /content/drive/MyDrive/AF/files
   Subdirectories: old

📄 File Types Found:
   .atr: 50 files
   .atr-: 2 files
   .dat: 23 files
   .hea: 25 files
   .hea-: 23 files
   .qrs: 25 files
   .qrs-: 1 files
   .qrsc: 2 files
   .txt: 2 files
   .xws: 1 files

📊 Metadata Files:
   - SHA256SUMS.txt
   - notes.txt

🔍 Detected Format: WFDB
   Total record files: 23


✅ Found 25 records in WFDB format

🔍 Validating 25 records...
   Progress: 10/25 (8 valid)
   Progress: 20/25 (18 valid)

✅ Found 23 valid records
⚠️  Skipped 2 problematic records:
   • 00735: Failed to load record '00735': Error reading WFDB file: samp...
   • 03665: Failed to load record '03665': Error reading WFDB file: samp...
Valid records: ['04015', '04043', '04048', '04126', '04746', '04908', '04936', '05091', '05121', '05261', '06426', '06453', '06995', '07162', '07859', '07879', '07910', '08215', '08219', '08378', '08405', '08434', '0

In [8]:
import wfdb
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple

class UniversalAnnotationAnalyzer:
    """
    Label-agnostic annotation analyzer - discovers all labels automatically.
    No hardcoded patterns or assumptions about label names.
    """

    def __init__(self, loader: 'UniversalECGLoader', annotation_extension: str = 'atr'):
        self.loader = loader
        self.data_path = loader.data_path
        self.annotation_ext = annotation_extension
        self.records_with_annotations = self._discover_annotated_records()

    def _discover_annotated_records(self) -> List[str]:
        """Find records with annotation files"""
        all_records = self.loader.get_valid_records()
        annotated = [r for r in all_records
                    if (self.data_path / f"{r}.{self.annotation_ext}").exists()]
        return annotated

    def get_annotations(self, record_name: str) -> Optional[wfdb.Annotation]:
        """Load annotations for a record"""
        try:
            return wfdb.rdann(str(self.data_path / record_name), self.annotation_ext)
        except Exception as e:
            return None

    def analyze_single_record(self, record_name: str) -> Optional[Dict]:
        """Analyze annotations for one record"""
        ann = self.get_annotations(record_name)
        if ann is None:
            return None

        try:
            signal_data = self.loader.load_record(record_name)
            fs = signal_data['fs']
            total_samples = signal_data['signal'].shape[0]
            duration_sec = signal_data['duration']
        except:
            return None

        # Extract rhythm annotations (aux_note contains rhythm labels)
        rhythms = [note.strip() for note in ann.aux_note if note.strip()]
        rhythm_samples = [ann.sample[i] for i, note in enumerate(ann.aux_note) if note.strip()]

        if not rhythms:
            return None

        # Calculate durations and create segments
        rhythm_durations = {}
        rhythm_segments = []

        for i in range(len(rhythms)):
            rhythm = rhythms[i]
            start_sample = rhythm_samples[i]
            end_sample = rhythm_samples[i + 1] if i < len(rhythms) - 1 else total_samples
            duration = (end_sample - start_sample) / fs

            rhythm_durations[rhythm] = rhythm_durations.get(rhythm, 0) + duration

            rhythm_segments.append({
                'rhythm': rhythm,
                'start_sample': start_sample,
                'end_sample': end_sample,
                'start_time': start_sample / fs,
                'end_time': end_sample / fs,
                'duration_sec': duration
            })

        return {
            'record': record_name,
            'fs': fs,
            'total_duration': duration_sec,
            'rhythm_counts': Counter(rhythms),
            'rhythm_durations': rhythm_durations,
            'rhythm_segments': rhythm_segments
        }

    def analyze_all_records(self, max_records: Optional[int] = None) -> Tuple[List[Dict], Dict]:
        """Analyze all annotated records"""
        records = self.records_with_annotations[:max_records] if max_records else self.records_with_annotations

        if not records:
            print("No annotated records found")
            return [], {}

        print(f"Analyzing {len(records)} records...")

        all_results = []
        all_rhythm_counts = Counter()
        all_rhythm_durations = Counter()
        total_duration = 0

        for record in records:
            result = self.analyze_single_record(record)
            if result:
                all_results.append(result)
                all_rhythm_counts.update(result['rhythm_counts'])
                total_duration += result['total_duration']
                for rhythm, duration in result['rhythm_durations'].items():
                    all_rhythm_durations[rhythm] += duration

        summary = {
            'total_records': len(all_results),
            'total_duration_hours': total_duration / 3600,
            'all_rhythms': dict(all_rhythm_counts),
            'rhythm_durations_hours': {r: d/3600 for r, d in all_rhythm_durations.items()}
        }

        # Print clean summary
        print(f"\nDataset Summary:")
        print(f"  Records analyzed: {summary['total_records']}")
        print(f"  Total duration: {summary['total_duration_hours']:.2f} hours")
        print(f"\nRhythm labels found:")
        for rhythm in sorted(all_rhythm_counts.keys()):
            duration_hrs = all_rhythm_durations[rhythm] / 3600
            percentage = (all_rhythm_durations[rhythm] / total_duration) * 100
            print(f"  {rhythm}: {duration_hrs:.2f}h ({percentage:.1f}%)")

        return all_results, summary

    def get_all_unique_labels(self, all_results: List[Dict]) -> List[str]:
        """Extract all unique rhythm labels found in dataset"""
        all_labels = set()
        for result in all_results:
            all_labels.update(result['rhythm_counts'].keys())
        return sorted(list(all_labels))

    def export_to_csv(self, all_results: List[Dict], output_path: str = 'annotations.csv') -> pd.DataFrame:
        """Export detailed analysis to CSV"""
        rows = []
        for result in all_results:
            for rhythm, duration in result['rhythm_durations'].items():
                rows.append({
                    'record': result['record'],
                    'rhythm_label': rhythm,
                    'duration_sec': duration,
                    'duration_hours': duration / 3600,
                    'occurrences': result['rhythm_counts'][rhythm],
                    'sampling_rate': result['fs']
                })

        df = pd.DataFrame(rows)
        df.to_csv(output_path, index=False)
        print(f"\nExported to: {output_path}")
        return df


class RhythmMapper:
    """
    Maps discovered rhythm labels to training classes.
    User defines the mapping based on what they found in the data.
    """

    def __init__(self, label_mapping: Dict[str, int], default_class: Optional[int] = None):
        """
        Args:
            label_mapping: Dict mapping rhythm labels to class indices
                          e.g., {'(AFIB': 1, '(AFL': 1, '(N': 0}
            default_class: Class to assign to unmapped labels (None = exclude)
        """
        self.label_mapping = label_mapping
        self.default_class = default_class

    def map_label(self, rhythm_label: str) -> Optional[int]:
        """Map a rhythm label to a class index"""
        return self.label_mapping.get(rhythm_label, self.default_class)

    def get_mapped_labels(self) -> List[str]:
        """Get list of labels that are mapped"""
        return list(self.label_mapping.keys())

    def get_unmapped_labels(self, all_labels: List[str]) -> List[str]:
        """Get labels that are not in the mapping"""
        return [label for label in all_labels if label not in self.label_mapping]

    def print_mapping(self):
        """Display the current mapping"""
        print("Rhythm Label Mapping:")
        for label, class_idx in sorted(self.label_mapping.items(), key=lambda x: x[1]):
            print(f"  {label} → Class {class_idx}")
        if self.default_class is not None:
            print(f"  [unmapped] → Class {self.default_class}")
        else:
            print(f"  [unmapped] → Excluded")

In [ ]:

# ============================================================
# USAGE EXAMPLE
# ============================================================

# Step 1: Initialize loader and analyzer
loader = UniversalECGLoader('/content/drive/MyDrive/AF/files')
analyzer = UniversalAnnotationAnalyzer(loader, annotation_extension='atr')

print(f"Found {len(analyzer.records_with_annotations)} annotated records\n")

# Step 2: Analyze dataset to discover all labels
all_results, summary = analyzer.analyze_all_records()

# Step 3: Get all unique labels (NO HARDCODING - pure discovery)
unique_labels = analyzer.get_all_unique_labels(all_results)
print(f"\nAll unique rhythm labels in dataset:")
for label in unique_labels:
    print(f"  - {label}")

# Step 4: Export for detailed inspection
df = analyzer.export_to_csv(all_results, 'rhythm_analysis.csv')

# Step 5: Create your own mapping based on discovered labels
print("\n" + "="*60)
print("Next: Define your RhythmMapper based on labels above")
print("="*60)
print("\nExample:")
print("mapper = RhythmMapper(")
print("    label_mapping={")
print("        '(AFIB': 1,  # Your discovered AFib label")
print("        '(N': 0,     # Your discovered Normal label")
print("    },")
print("    default_class=None  # Exclude unmapped labels")
print(")")


ANALYZING DATASET AT: /content/drive/MyDrive/AF/files

📁 Directory Structure:
   Root: /content/drive/MyDrive/AF/files
   Subdirectories: old

📄 File Types Found:
   .atr: 50 files
   .atr-: 2 files
   .dat: 23 files
   .hea: 25 files
   .hea-: 23 files
   .qrs: 25 files
   .qrs-: 1 files
   .qrsc: 2 files
   .txt: 2 files
   .xws: 1 files

📊 Metadata Files:
   - SHA256SUMS.txt
   - notes.txt

🔍 Detected Format: WFDB
   Total record files: 23


✅ Found 25 records in WFDB format

🔍 Validating 25 records...
   Progress: 10/25 (8 valid)
   Progress: 20/25 (18 valid)

✅ Found 23 valid records
⚠️  Skipped 2 problematic records:
   • 00735: Failed to load record '00735': Error reading WFDB file: samp...
   • 03665: Failed to load record '03665': Error reading WFDB file: samp...
Found 23 annotated records

Analyzing 23 records...
